# Backtest tour — dynamic-selection walk-forward evaluation

Visualisation layer over the saved outputs of the **dynamic-selection walk-forward
backtest** (`scripts/run_backtest.py` → `data/processed/backtest/`). No selection or
backtest logic runs here — regenerate the artifacts with:

    conda run -n ip2 python scripts/build_processed.py   # processed panels (once)
    conda run -n ip2 python scripts/run_backtest.py      # the walk-forward run

Per re-selection regime (every 2 years; P&L 2006–2025 on an expanding window;
1998–1999 loadings warm-up, 2000–2005 initial training), the driver
(`src/backtest/dynamic.py`):

1. **selects** the factor set point-in-time at the regime's cutoff — three nested
   filters (single-factor IR/FM gate → correlation-cluster representatives → lasso) on
   data sliced to `period ≤ cutoff` (see `pipeline_tour.ipynb` §3–4);
2. **re-estimates beliefs** on the selected set — pooled FM premia, lag-1 ICs and the
   monthly risk-model histories are rebuilt per regime (`src/backtest/inputs.py`);
3. **runs the engine segment** over the regime's formation quarters with the previous
   segment's book and risk-EMA state threaded through: turnover is charged against the
   carried drifted book, and factor EMAs align by name across shortlist switches.

Engine row convention: the book formed at `period` t realizes its P&L over t+1
(`pnl_period`). Costs hit voluntary turnover only; delisting exits settle without a
trade.

In [ ]:
%load_ext autoreload
%autoreload 2

import json, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
from plotnine import (ggplot, aes, geom_col, geom_line, geom_point, geom_hline,
                      geom_vline, geom_tile, coord_flip, facet_wrap,
                      scale_fill_manual, labs, theme_bw, theme, element_text)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.backtest.metrics import (factor_health, max_drawdown, regime_performance,
                                  selection_frequency, selection_transitions,
                                  sharpe_ratio, shortlist_turnover)

cfg = config.load(PROJECT_ROOT / "config.yaml")
BT = PROJECT_ROOT / cfg["data"]["cache"] / "backtest"
pl.Config.set_tbl_rows(35)

results = pl.read_ipc(BT / "results.feather")            # one row per traded rebalance (+ regime)
weights = pl.read_ipc(BT / "weights.feather")             # [period, stock_id, weight, regime]
history = pl.read_ipc(BT / "selection_history.feather")   # stacked per-cutoff scorecards
perf    = pl.read_ipc(BT / "performance.feather")         # results + mkt/bench + hedged
meta    = json.loads((BT / "run_meta.json").read_text())

print(f"run {meta['built_at']} | {len(meta['regimes'])} regimes | "
      f"{results.height} traded rebalances | schedule {meta['schedule']}")
for d in meta["diagnostics"]:
    if "n_selected" in d:
        flags = " CARRIED" if d["carried_forward"] else ""
        flags += " BELOW-MIN" if d["below_min_warn"] else ""
        print(f"  regime {d['regime']} cutoff {d['cutoff']}: {d['n_selected']} factors{flags}")

## 1. Headline metrics

The strategy's broad-market beta is hedged externally by design
(`backtest.hedge_broad_market_beta` — an accounting-only, costless overlay), so
performance is always evaluated on the **hedged** series; the raw net series rides
along for comparison. Both are quarterly, excess-of-RF and net of L1 transaction
costs. Book/cost diagnostics (and the IR vs the cap-weighted tradeable-universe
benchmark) come from the run script's saved summary.

In [ ]:
def evaluate(col):
    r = perf[col].to_numpy()
    n = len(r)
    return {"ann_return": float(np.prod(1.0 + r) ** (4.0 / n) - 1.0),
            "ann_vol": float(np.std(r, ddof=1) * 2.0),
            "sharpe": sharpe_ratio(r, periods_per_year=4),
            "max_drawdown": max_drawdown(r)}

net, hedged = evaluate("net_ret"), evaluate("hedged")
book = {k: meta["summary"][k] for k in
        ("quarters_traded", "ir_vs_benchmark", "avg_turnover", "avg_tc_bps",
         "avg_gross_leverage", "avg_net_exposure", "avg_mkt_beta", "return_data_gaps")}
print(pl.DataFrame({"metric": list(book),
                    "value": [float(round(v, 4)) for v in book.values()]}))
pl.DataFrame({"metric": list(hedged),
              "hedged": [round(v, 4) for v in hedged.values()],
              "net (unhedged)": [round(v, 4) for v in net.values()]})

## 2. Equity curves & drawdown

Aligned on the realization period (`pnl_period`): row `t` books the P&L of the book
formed at `t` over `t+1`. Dotted verticals mark the re-selection regime boundaries —
the quarters where the factor set (and the premia beliefs behind μ) switch. Drawdown
is evaluated on the hedged series.

In [ ]:
series = {"net_ret": "strategy (net)", "hedged": "strategy (net, hedged)",
          "bench": "cap-weighted universe"}
cum = (perf.sort("pnl_period")
       .with_columns(pl.col("bench").fill_null(0.0))
       .with_columns([((pl.col(c) + 1.0).cum_prod() - 1.0).alias(lbl)
                      for c, lbl in series.items()])
       .unpivot(index="pnl_period", on=list(series.values()),
                variable_name="series", value_name="cum_ret")
       .to_pandas())
regime_starts = (perf.group_by("regime").agg(start=pl.col("pnl_period").min())
                 .sort("regime")["start"].cast(pl.Datetime).to_list())
(ggplot(cum, aes("pnl_period", "cum_ret", color="series"))
 + geom_vline(xintercept=regime_starts, linetype="dotted", alpha=0.6)
 + geom_line()
 + geom_hline(yintercept=0, linetype="dashed", alpha=0.5)
 + labs(title="Cumulative performance (quarterly compounding, net of costs)",
        subtitle="dotted verticals = re-selection regime boundaries",
        x="", y="cumulative excess return", color="")
 + theme_bw() + theme(figure_size=(10, 5), legend_position="top"))

In [ ]:
dd = (perf.sort("pnl_period")
      .with_columns(nav=(pl.col("hedged") + 1.0).cum_prod())
      .with_columns(drawdown=pl.col("nav") / pl.col("nav").cum_max() - 1.0)
      .select("pnl_period", "drawdown", "hedged").to_pandas())
(ggplot(dd, aes("pnl_period"))
 + geom_col(aes(y="hedged"), fill="#2c7fb8", alpha=0.7)
 + geom_line(aes(y="drawdown"), color="#d95f02")
 + geom_hline(yintercept=0, size=0.3)
 + labs(title="Quarterly hedged net returns (bars) and drawdown (line)",
        x="", y="quarterly hedged net return / drawdown")
 + theme_bw() + theme(figure_size=(10, 4)))

## 3. Book diagnostics

Leverage, exposure, ex-ante market beta, turnover, cost and breadth per rebalance —
straight from `performance.feather`.

In [ ]:
diag = (perf.with_columns(tc_bps=pl.col("tc") * 1e4)
        .select("pnl_period", "gross_lev", "net_exp", "mkt_beta", "turnover", "tc_bps",
                "n_long", "n_short", "n_names")
        .unpivot(index="pnl_period", variable_name="metric", value_name="value")
        .to_pandas())
(ggplot(diag, aes("pnl_period", "value"))
 + geom_line()
 + facet_wrap("~metric", ncol=3, scales="free_y")
 + labs(title="Book diagnostics per rebalance", x="", y="")
 + theme_bw()
 + theme(figure_size=(11, 7), subplots_adjust={"hspace": 0.45, "wspace": 0.3},
         strip_text=element_text(size=9), axis_text=element_text(size=7)))

## 4. Selection dynamics

`selection_history.feather` stacks the full per-cutoff scorecards (every factor ×
every re-selection window: gates, cluster, lasso, selection), and
`src/backtest/metrics.py` provides pure-function summaries over it — membership
(`selection_frequency`), start/stop events (`selection_transitions`), stability
(`shortlist_turnover`) and the gate-statistic trajectories (`factor_health`).

In [ ]:
freq = selection_frequency(history)
order = freq.sort("freq", "factor", descending=[False, True])["factor"].to_list()
hm = (history.with_columns(
          cutoff=pl.format("{}Q{}", pl.col("cutoff").dt.year(), pl.col("cutoff").dt.quarter()),
          factor=pl.col("factor").cast(pl.Enum(order)),
          state=pl.when(pl.col("selected")).then(pl.lit("selected"))
                 .when(pl.col("single_pass")).then(pl.lit("gate pass, not selected"))
                 .otherwise(pl.lit("not selected")))
      .to_pandas())
(ggplot(hm, aes("cutoff", "factor", fill="state"))
 + geom_tile(color="white")
 + scale_fill_manual(values={"selected": "#2c7fb8",
                             "gate pass, not selected": "#a6bddb",
                             "not selected": "#efefef"})
 + labs(title="Factor x re-selection window membership", x="selection cutoff", y="", fill="")
 + theme_bw()
 + theme(figure_size=(9, 9), axis_text_x=element_text(rotation=45, hjust=1),
         axis_text_y=element_text(size=7), legend_position="top"))

In [ ]:
fqp = freq.to_pandas()
(ggplot(fqp, aes("reorder(factor, freq)", "freq", fill="current"))
 + geom_col() + coord_flip()
 + labs(title=f"Selection frequency across the {history['cutoff'].n_unique()} windows",
        x="", y="share of windows selected", fill="in current shortlist")
 + theme_bw() + theme(figure_size=(8, 8)))

### Gate-statistic trajectories — factors starting / stopping to work

Each re-selection re-tests every factor on the expanded window, so the stacked
scorecards trace a natural "is this factor still alive" panel. Lines show the
expanding-window statistic per cutoff for every factor that was **ever selected**;
dashed lines mark the gate; orange markers are the shortlist entry/exit events from
`selection_transitions`.

In [ ]:
ever = (history.group_by("factor").agg(ever=pl.col("selected").any())
        .filter(pl.col("ever"))["factor"].to_list())
health = factor_health(history).filter(
    pl.col("factor").is_in(ever) & pl.col("metric").is_in(["ir_lag1", "fm_t"])
)
events = selection_transitions(history)
hp = health.join(events, on=["factor", "cutoff"], how="left").to_pandas()

def health_plot(metric, gate, title):
    d = hp[hp.metric == metric]
    ev = d[d.event.notna()]
    return (
        ggplot(d, aes("cutoff", "value"))
        + geom_hline(yintercept=[-gate, gate], linetype="dashed", alpha=0.5)
        + geom_line(color="#999999")
        + geom_point(aes(color="selected"), size=1.2)
        + geom_point(data=ev, mapping=aes(shape="event"), size=2.5, color="#d95f02")
        + facet_wrap("~factor", ncol=5)
        + labs(title=title, x="selection cutoff", y=metric, shape="")
        + theme_bw()
        + theme(figure_size=(13, 10), subplots_adjust={"hspace": 0.6, "wspace": 0.25},
                strip_text=element_text(size=7), axis_text=element_text(size=6),
                axis_text_x=element_text(rotation=45, hjust=1), legend_position="top")
    )

health_plot("ir_lag1", float(cfg["validation"]["ic"]["ir_shortlist_threshold"]),
            "Lag-1 IC-IR across re-selection windows (ever-selected factors)")

In [ ]:
health_plot("fm_t", float(cfg["validation"]["fama_macbeth"]["t_stat_threshold"]),
            "Pooled Fama-MacBeth t-stat across re-selection windows (ever-selected factors)")

### Shortlist stability

Adds/drops and Jaccard similarity between consecutive shortlists, plus the shortlist
size against the soft floor (`validation.selection.min_shortlist_warn` — a warning
metric, never a hard constraint).

In [ ]:
to = shortlist_turnover(history)
print(to)
min_warn = int(cfg["validation"]["selection"]["min_shortlist_warn"])
top = (to.with_columns(cutoff=pl.format("{}Q{}", pl.col("cutoff").dt.year(),
                                        pl.col("cutoff").dt.quarter()))
       .unpivot(index="cutoff", on=["n_selected", "n_added", "n_dropped", "jaccard_prev"],
                variable_name="metric", value_name="value")
       .drop_nulls("value").to_pandas())
warn = pl.DataFrame({"metric": ["n_selected"], "y": [float(min_warn)]}).to_pandas()
(ggplot(top, aes("cutoff", "value"))
 + geom_col(fill="#2c7fb8")
 + geom_hline(data=warn, mapping=aes(yintercept="y"), linetype="dashed", color="#d95f02")
 + facet_wrap("~metric", ncol=2, scales="free_y")
 + labs(title="Shortlist size and turnover between consecutive selections",
        subtitle=f"dashed = soft floor ({min_warn}); jaccard_prev: 1 = identical, 0 = full replacement",
        x="selection cutoff", y="")
 + theme_bw()
 + theme(figure_size=(10, 6), axis_text_x=element_text(rotation=45, hjust=1)))

## 5. Per-regime performance — did re-selection help?

`regime_performance` slices the traded quarters by regime, evaluated on the **hedged**
series (the strategy's actual design; the `net_ret` column of the input is swapped for
`hedged`); joined with each regime's selection diagnostics (shortlist size,
carried-forward flag) from `run_meta.json`.

In [ ]:
sel_diag = pl.DataFrame([
    {"regime": d["regime"], "cutoff": d["cutoff"], "n_selected": d["n_selected"],
     "carried_forward": d["carried_forward"]}
    for d in meta["diagnostics"] if "n_selected" in d
])
rp = (regime_performance(perf.with_columns(net_ret=pl.col("hedged")))
      .rename({"ann_ret": "ann_ret_hedged", "sharpe": "sharpe_hedged",
               "max_drawdown": "max_drawdown_hedged"})
      .join(sel_diag, on="regime", how="left")
      .select("regime", "cutoff", "n_selected", "carried_forward", "start", "end",
              "n_periods", "ann_ret_hedged", "sharpe_hedged", "max_drawdown_hedged",
              "avg_turnover", "avg_tc"))
rp

In [ ]:
rpp = rp.with_columns(positive=pl.col("sharpe_hedged") > 0).to_pandas()
(ggplot(rpp, aes("factor(regime)", "sharpe_hedged", fill="positive"))
 + geom_col()
 + geom_hline(yintercept=0, size=0.3)
 + labs(title="Per-regime Sharpe (hedged)", x="regime (re-selection window)",
        y="annualized Sharpe")
 + theme_bw() + theme(figure_size=(9, 4), legend_position="none"))

## Caveats

* **Selection is point-in-time per regime** (the old hardcoded `SHORTLIST` is gone):
  each cutoff's statistics use only `period ≤ cutoff`. An empty selection would carry
  the previous shortlist forward (flagged `carried_forward`); shortlists below the soft
  floor are flagged, never enforced.
* **Realized P&L is unwinsorized by design** (`ret_raw` — a real book eats real
  prints, including multi-hundred-percent squeezes on shorts). Hence the
  `portfolio.constraints.max_name_weight` concentration cap; the engine additionally
  restarts from cash if NAV is ever wiped out (`book_wiped` in the diagnostics).
* **Costs** hit voluntary turnover only; involuntary delisting exits settle without a
  trade (wipeout losses/gains arrive through the terminal return instead).
* **Hedging** of the broad-market beta is accounting-only and assumed costless (the
  `hedged` series); the optimizer never enforces beta neutrality — sector beta stays
  in the book.
* **Solvers**: CLARABEL is blocked by this machine's Application Control policy; the
  run script uses OSQP first (tight tolerances) with SCS/HIGHS retries and
  sanity-checks every solution. Skips, book-wipes and solver fallbacks are recorded in
  `run_meta.json`'s diagnostics.